# Predictive Price Engine

This notebook trains a Machine Learning model (Random Forest Regressor) to predict property values (`AMOUNT`) in Dubai based on historical transactions.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

plt.style.use('ggplot')
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', None)

## 1. Data Loading & Cleaning

In [2]:
data_dir = 'data'
df = pd.read_csv(os.path.join(data_dir, 'transactions-2026-03-04.csv'))
print(f"Initial data shape: {df.shape}")
print(df.dtypes)

Initial data shape: (45308, 22)
TRANSACTION_NUMBER       object
TRANSACTION_DATE         object
TRANSACTION_TYPE         object
TRANSACTION_SUB_TYPE     object
REGISTRATION_TYPE        object
IS_FREE_HOLD ?           object
USAGE                    object
AREA                     object
PROPERTY_TYPE            object
PROPERTY_SUB_TYPE        object
AMOUNT                  float64
TRANSACTION_SIZE        float64
PROPERTY_SIZE           float64
ROOMS                    object
PARKING                  object
NEAREST_METRO            object
NEAREST_MALL             object
NEAREST_LANDMARK         object
TOTAL_BUYER               int64
TOTAL_SELLER              int64
MASTER_PROJECT           object
PROJECT                  object
dtype: object


In [3]:
    
unqiue_cols = ['TRANSACTION_NUMBER']
datetime_cols= ['TRANSACTION_DATE']
cat_cols= ['TRANSACTION_TYPE', 'TRANSACTION_SUB_TYPE', 'REGISTRATION_TYPE', 'IS_FREE_HOLD ?', 
            'USAGE','AREA', 'PROPERTY_TYPE', 'PROPERTY_SUB_TYPE','ROOMS','PARKING', 'NEAREST_METRO', 'NEAREST_MALL', 'NEAREST_LANDMARK',
                'MASTER_PROJECT', 'PROJECT' ]
num_cols= [ 'AMOUNT','TRANSACTION_SIZE', 'PROPERTY_SIZE', 'TOTAL_BUYER', 'TOTAL_SELLER']

# 1. Convert columns to float
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype(float)
    
# 2. Convert to datetime
for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')
    
# 3. Convert categorical text columns to 'category' datatype
for col in cat_cols:
    df[col] = df[col].astype('category')


In [4]:
print(df.dtypes)

TRANSACTION_NUMBER              object
TRANSACTION_DATE        datetime64[ns]
TRANSACTION_TYPE              category
TRANSACTION_SUB_TYPE          category
REGISTRATION_TYPE             category
IS_FREE_HOLD ?                category
USAGE                         category
AREA                          category
PROPERTY_TYPE                 category
PROPERTY_SUB_TYPE             category
AMOUNT                         float64
TRANSACTION_SIZE               float64
PROPERTY_SIZE                  float64
ROOMS                         category
PARKING                       category
NEAREST_METRO                 category
NEAREST_MALL                  category
NEAREST_LANDMARK              category
TOTAL_BUYER                    float64
TOTAL_SELLER                   float64
MASTER_PROJECT                category
PROJECT                       category
dtype: object


In [5]:
df.nunique()

TRANSACTION_NUMBER      44242
TRANSACTION_DATE        43387
TRANSACTION_TYPE            3
TRANSACTION_SUB_TYPE       31
REGISTRATION_TYPE           2
IS_FREE_HOLD ?              2
USAGE                       2
AREA                      248
PROPERTY_TYPE               3
PROPERTY_SUB_TYPE          30
AMOUNT                  20984
TRANSACTION_SIZE        15759
PROPERTY_SIZE           15589
ROOMS                      12
PARKING                  6482
NEAREST_METRO              53
NEAREST_MALL                5
NEAREST_LANDMARK           13
TOTAL_BUYER                 1
TOTAL_SELLER                1
MASTER_PROJECT              8
PROJECT                  2549
dtype: int64

In [6]:
# Create a mask for all duplicate rows
duplicate_mask = df.duplicated(keep=False)

# Filter the dataframe to show only the duplicates
duplicates = df[duplicate_mask]

# Sort by a column (like AMOUNT or TRANSACTION_DATE) to see the identical rows grouped together
duplicates_sorted = duplicates.sort_values(by=list(df.columns))

# Display the duplicate rows
display(duplicates_sorted)


,TRANSACTION_NUMBER,TRANSACTION_DATE,TRANSACTION_TYPE,TRANSACTION_SUB_TYPE,REGISTRATION_TYPE,IS_FREE_HOLD ?,USAGE,AREA,PROPERTY_TYPE,PROPERTY_SUB_TYPE,AMOUNT,TRANSACTION_SIZE,PROPERTY_SIZE,ROOMS,PARKING,NEAREST_METRO,NEAREST_MALL,NEAREST_LANDMARK,TOTAL_BUYER,TOTAL_SELLER,MASTER_PROJECT,PROJECT
42419,43-20-2026,2026-01-09 11:04:31,Mortgage,Portfolio Mortgage Registration,Ready,Free Hold,Residential,DUBAI MARINA,Unit,Flat,1530000.00,50.63,50.63,Studio,NaN,Marina Mall Metro Station,Marina Mall,Burj Al Arab,0.0,0.0,NaN,DUBAI MARINA MALL
42420,43-20-2026,2026-01-09 11:04:31,Mortgage,Portfolio Mortgage Registration,Ready,Free Hold,Residential,DUBAI MARINA,Unit,Flat,1530000.00,50.63,50.63,Studio,NaN,Marina Mall Metro Station,Marina Mall,Burj Al Arab,0.0,0.0,NaN,DUBAI MARINA MALL
42537,43-29-2026,2026-01-23 08:33:40,Mortgage,Portfolio Mortgage Registration,Ready,Free Hold,Residential,DISCOVERY GARDENS,Unit,Flat,535158.56,147.18,147.18,2 B/R,NaN,Harbour Tower,Ibn-e-Battuta Mall,Sports City Swimming Academy,0.0,0.0,NaN,NaN
42539,43-29-2026,2026-01-23 08:33:40,Mortgage,Portfolio Mortgage Registration,Ready,Free Hold,Residential,DISCOVERY GARDENS,Unit,Flat,535158.56,147.18,147.18,2 B/R,NaN,Harbour Tower,Ibn-e-Battuta Mall,Sports City Swimming Academy,0.0,0.0,NaN,NaN
42551,43-30-2026,2026-01-23 09:57:13,Mortgage,Portfolio Mortgage Registration,Ready,Free Hold,Residential,DISCOVERY GARDENS,Unit,Flat,566462.90,146.90,146.90,2 B/R,NaN,Harbour Tower,Ibn-e-Battuta Mall,Sports City Swimming Academy,0.0,0.0,NaN,NaN
42553,43-30-2026,2026-01-23 09:57:13,Mortgage,Portfolio Mortgage Registration,Ready,Free Hold,Residential,DISCOVERY GARDENS,Unit,Flat,566462.90,146.90,146.90,2 B/R,NaN,Harbour Tower,Ibn-e-Battuta Mall,Sports City Swimming Academy,0.0,0.0,NaN,NaN
42554,43-31-2026,2026-01-23 10:01:52,Mortgage,Portfolio Mortgage Registration,Ready,Free Hold,Residential,DISCOVERY GARDENS,Unit,Flat,573976.80,147.44,147.44,2 B/R,NaN,Harbour Tower,Ibn-e-Battuta Mall,Sports City Swimming Academy,0.0,0.0,NaN,NaN
42555,43-31-2026,2026-01-23 10:01:52,Mortgage,Portfolio Mortgage Registration,Ready,Free Hold,Residential,DISCOVERY GARDENS,Unit,Flat,573976.80,147.44,147.44,2 B/R,NaN,Harbour Tower,Ibn-e-Battuta Mall,Sports City Swimming Academy,0.0,0.0,NaN,NaN
42563,43-31-2026,2026-01-23 10:01:52,Mortgage,Portfolio Mortgage Registration,Ready,Free Hold,Residential,DISCOVERY GARDENS,Unit,Flat,573976.80,147.44,147.44,2 B/R,NaN,Harbour Tower,Ibn-e-Battuta Mall,Sports City Swimming Academy,0.0,0.0,NaN,NaN
42576,43-34-2026,2026-01-28 15:07:10,Mortgage,Portfolio Mortgage Registration,Ready,Free Hold,Residential,PALM JUMEIRAH,Unit,Flat,4540087.50,106.91,106.91,1 B/R,NaN,Knowledge Village,Marina Mall,Burj Al Arab,0.0,0.0,NaN,AZURE RESIDENCE


In [7]:
duplicates = df[df.duplicated(keep=False)]
print(duplicates.shape)

(60, 22)


In [8]:
df = df.drop_duplicates()

In [9]:
df.shape

(45270, 22)

In [10]:
df.nunique()

TRANSACTION_NUMBER      44242
TRANSACTION_DATE        43387
TRANSACTION_TYPE            3
TRANSACTION_SUB_TYPE       31
REGISTRATION_TYPE           2
IS_FREE_HOLD ?              2
USAGE                       2
AREA                      248
PROPERTY_TYPE               3
PROPERTY_SUB_TYPE          30
AMOUNT                  20984
TRANSACTION_SIZE        15759
PROPERTY_SIZE           15589
ROOMS                      12
PARKING                  6482
NEAREST_METRO              53
NEAREST_MALL                5
NEAREST_LANDMARK           13
TOTAL_BUYER                 1
TOTAL_SELLER                1
MASTER_PROJECT              8
PROJECT                  2549
dtype: int64

In [11]:
duplicates = df[df.duplicated(keep=False)]
print(duplicates.shape)

(0, 22)


In [12]:
df.dtypes

TRANSACTION_NUMBER              object
TRANSACTION_DATE        datetime64[ns]
TRANSACTION_TYPE              category
TRANSACTION_SUB_TYPE          category
REGISTRATION_TYPE             category
IS_FREE_HOLD ?                category
USAGE                         category
AREA                          category
PROPERTY_TYPE                 category
PROPERTY_SUB_TYPE             category
AMOUNT                         float64
TRANSACTION_SIZE               float64
PROPERTY_SIZE                  float64
ROOMS                         category
PARKING                       category
NEAREST_METRO                 category
NEAREST_MALL                  category
NEAREST_LANDMARK              category
TOTAL_BUYER                    float64
TOTAL_SELLER                   float64
MASTER_PROJECT                category
PROJECT                       category
dtype: object

In [13]:
df.isna().sum()

TRANSACTION_NUMBER          0
TRANSACTION_DATE            0
TRANSACTION_TYPE            0
TRANSACTION_SUB_TYPE        0
REGISTRATION_TYPE           0
IS_FREE_HOLD ?              0
USAGE                       0
AREA                        0
PROPERTY_TYPE               0
PROPERTY_SUB_TYPE         737
AMOUNT                      0
TRANSACTION_SIZE            1
PROPERTY_SIZE               0
ROOMS                    7015
PARKING                 11488
NEAREST_METRO           21728
NEAREST_MALL            22314
NEAREST_LANDMARK        16294
TOTAL_BUYER                 0
TOTAL_SELLER                0
MASTER_PROJECT          45110
PROJECT                  6189
dtype: int64

#Analysis of DATA

In [14]:
import pygwalker as pyg

# use_kernel_calc=True tells PyGWalker to use an internal database 
# engine (DuckDB) which is much faster for large datasets!
walker = pyg.walk(df, use_kernel_calc=True)



Box(children=(HTML(value='\n<div id="ifr-pyg-00064c2f9fa2cb91i8ouqxr6Q4Ig5ROV" style="height: auto">\n    <hea…

In [15]:

duplicate_transaction_numbers = df[df.duplicated(subset=['TRANSACTION_NUMBER'], keep= 'first')]
duplicate_transaction_numbers_sorted = duplicate_transaction_numbers.sort_values(by=['TRANSACTION_NUMBER'])
display(duplicate_transaction_numbers_sorted)



,TRANSACTION_NUMBER,TRANSACTION_DATE,TRANSACTION_TYPE,TRANSACTION_SUB_TYPE,REGISTRATION_TYPE,IS_FREE_HOLD ?,USAGE,AREA,PROPERTY_TYPE,PROPERTY_SUB_TYPE,AMOUNT,TRANSACTION_SIZE,PROPERTY_SIZE,ROOMS,PARKING,NEAREST_METRO,NEAREST_MALL,NEAREST_LANDMARK,TOTAL_BUYER,TOTAL_SELLER,MASTER_PROJECT,PROJECT
1,101-2-2026,2026-01-29 18:04:38,Mortgage,Portfolio Mortgage Registration Pre-Registration,Off-Plan,Free Hold,Residential,BUSINESS BAY,Unit,Flat,15925000.0,572.59,572.59,4 B/R,2,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,Downtown Dubai,0.0,0.0,NaN,THE CRESTMARK
2,101-2-2026,2026-01-29 18:04:38,Mortgage,Portfolio Mortgage Registration Pre-Registration,Off-Plan,Free Hold,Residential,BUSINESS BAY,Unit,Flat,15925000.0,509.08,509.08,4 B/R,2,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,Downtown Dubai,0.0,0.0,NaN,THE CRESTMARK
3,101-2-2026,2026-01-29 18:04:38,Mortgage,Portfolio Mortgage Registration Pre-Registration,Off-Plan,Free Hold,Residential,BUSINESS BAY,Unit,Flat,15925000.0,261.39,261.39,3 B/R,2,Buj Khalifa Dubai Mall Metro Station,Dubai Mall,Downtown Dubai,0.0,0.0,NaN,THE CRESTMARK
22822,11-1381-2026,2026-01-12 15:35:45,Sales,Sale,Ready,Free Hold,Residential,DUBAI SPORTS CITY,Unit,Flat,480000.0,38.45,38.45,Studio,P-039,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,0.0,0.0,NaN,ELITE II SPORTS RESIDENCE
23322,11-1847-2026,2026-01-15 08:40:23,Sales,Sale,Ready,Free Hold,Residential,DUBAI SPORTS CITY,Unit,Flat,420000.0,38.25,38.25,Studio,G-007,Nakheel Metro Station,Marina Mall,Sports City Swimming Academy,0.0,0.0,NaN,ELITE II SPORTS RESIDENCE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43758,715-5-2026,2026-01-26 10:51:44,Sales,Delayed Sell Lease to Own Registration,Ready,Free Hold,Commercial,JUMEIRAH PARK,Land,Commercial,5500000.0,548.64,548.64,NaN,NaN,Harbour Tower,Ibn-e-Battuta Mall,Sports City Swimming Academy,0.0,0.0,NaN,NaN
43760,715-6-2026,2026-01-23 10:08:31,Sales,Delayed Sell Lease to Own Registration,Ready,Free Hold,Residential,DUBAI CREEK HARBOUR,Unit,Flat,2350000.0,65.01,65.01,1 B/R,B3-A146,Creek Metro Station,City Centre Mirdif,Dubai International Airport,0.0,0.0,NaN,Palace Residences - Dubai Creek Harbour
43762,715-7-2026,2026-01-26 10:52:16,Mortgage,Delayed Sell Lease to Own Registration,Ready,Free Hold,Residential,Madinat Al Mataar,Building,Villa,3727500.0,370.28,370.28,5 B/R,NaN,NaN,NaN,NaN,0.0,0.0,NaN,The Pulse- Beachfront
43764,715-8-2026,2026-01-30 10:08:42,Sales,Delayed Sell Lease to Own Registration,Ready,Free Hold,Residential,AL FURJAN,Unit,Flat,1775000.0,112.25,112.25,2 B/R,G-13,Ibn Battuta Metro Station,Ibn-e-Battuta Mall,Expo 2020 Site,0.0,0.0,NaN,AMALIA RESIDENCES


In [16]:
!pip install ydata-profiling


In [23]:
pip install ydata-profiling

Note: you may need to restart the kernel to use updated packages.


In [ ]:

import pandas as pd
from ydata_profiling import ProfileReport

df = pd.read_csv('data/transactions-2026-03-04.csv')

# Drop missing values in these columns to avoid Seaborn errors
df_plot = df[['AMOUNT', 'TRANSACTION_SIZE', 'PROPERTY_SIZE']].dropna()

# We might want to filter out extreme outliers so the plots are readable
df_plot = df_plot[df_plot['AMOUNT'] < df_plot['AMOUNT'].quantile(0.95)]
df_plot = df_plot[df_plot['PROPERTY_SIZE'] < df_plot['PROPERTY_SIZE'].quantile(0.95)]

# Generate the report inside the script and use it as data_dashboard.html
profile = ProfileReport(df_plot, title="Dubai Property Prices Report (Interactions)", explorative=True)

# Save to an HTML file to view in the browser
profile.to_file("data_dashboard.html")
print("Dashboard generated: data_dashboard.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 22/22 [00:00<00:00, 108.91it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
# Load data
data_dir = 'data'
df = pd.read_csv(os.path.join(data_dir, 'transactions-2026-03-04.csv'))
print(f"Initial data shape: {df.shape}")
print(df.dtypes)
# Convert metrics to numeric
df['AMOUNT'] = pd.to_numeric(df['AMOUNT'], errors='coerce')
df['PROPERTY_SIZE'] = pd.to_numeric(df['PROPERTY_SIZE'], errors='coerce')
# df = df.dropna(subset=['AMOUNT', 'PROPERTY_SIZE'])

print(f"Initial data shape: {df.shape}")
print(df.dtypes)

Initial data shape: (45308, 22)
TRANSACTION_NUMBER          str
TRANSACTION_DATE            str
TRANSACTION_TYPE            str
TRANSACTION_SUB_TYPE        str
REGISTRATION_TYPE           str
IS_FREE_HOLD ?              str
USAGE                       str
AREA                        str
PROPERTY_TYPE               str
PROPERTY_SUB_TYPE           str
AMOUNT                  float64
TRANSACTION_SIZE        float64
PROPERTY_SIZE           float64
ROOMS                       str
PARKING                     str
NEAREST_METRO               str
NEAREST_MALL                str
NEAREST_LANDMARK            str
TOTAL_BUYER               int64
TOTAL_SELLER              int64
MASTER_PROJECT              str
PROJECT                     str
dtype: object
Initial data shape: (45308, 22)
TRANSACTION_NUMBER          str
TRANSACTION_DATE            str
TRANSACTION_TYPE            str
TRANSACTION_SUB_TYPE        str
REGISTRATION_TYPE           str
IS_FREE_HOLD ?              str
USAGE                     

## 2. Outlier Handling

In [ ]:
# Filter out 1st and 99th percentiles for continuous variables to prevent skew
val_low = df['AMOUNT'].quantile(0.01)
val_high = df['AMOUNT'].quantile(0.99)

area_low = df['PROPERTY_SIZE'].quantile(0.01)
area_high = df['PROPERTY_SIZE'].quantile(0.99)

df_clean = df[(df['AMOUNT'] >= val_low) & (df['AMOUNT'] <= val_high) &
              (df['PROPERTY_SIZE'] >= area_low) & (df['PROPERTY_SIZE'] <= area_high)]

print(f"Cleaned data shape: {df_clean.shape}")

## 3. Feature Engineering & Encoding

In [ ]:
# Extract numeric rooms (e.g., '4 B/R' -> 4)
df_clean['ROOMS_NUM'] = df_clean['ROOMS'].str.extract('(\d+)').astype(float)
df_clean['ROOMS_NUM'] = df_clean['ROOMS_NUM'].fillna(1) # Assumption: missing means studio or 1 room

# Select relevant features for modeling
features = ['PROPERTY_SIZE', 'ROOMS_NUM', 'REGISTRATION_TYPE', 'IS_FREE_HOLD ?', 
            'USAGE', 'PROPERTY_TYPE', 'PROPERTY_SUB_TYPE']
target = 'AMOUNT'

X = df_clean[features]
y = df_clean[target]

# One-Hot Encode categorical variables
categorical_cols = X.select_dtypes(include=['object']).columns
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

## 4. Modeling & Validation

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Initialize Model (We use fewer estimators to speed up standard training)
model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)

# Cross Validation
cv_scores = cross_val_score(model, X_train, y_train, cv=3, scoring='r2')
print("Cross-Validation R² Scores:", cv_scores)
print(f"Mean CV R²: {cv_scores.mean():.4f}")

# Train Final Model
model.fit(X_train, y_train)

## 5. Benchmarking

In [ ]:
y_pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("--- Model Performance ---")
print(f"RMSE: {rmse:,.2f} AED")
print(f"MAE:  {mae:,.2f} AED")
print(f"R²:   {r2:.4f}")

## 6. Feature Importance

In [ ]:
importances = model.feature_importances_
feature_names = X_encoded.columns

# Create DataFrame for visualization
feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_imp_df, palette='viridis')
plt.title('Top 10 Feature Importances dictating Dubai Property Value')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()